# ERP003950: набор истины после фильтрации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Ветка предобработки: `results/ERP003950/`.

Биологический набор истины берётся только из её собственного `post_annotation_filtered/airr_pass`.

В ERP003950 нет UMI, поэтому точные V…J-последовательности дедуплицируются внутри каждого sample, а начальная численность по умолчанию равна `one_per_unique`. Наблюдаемая кратность сохраняется только для QC и анализа чувствительности.

**Фрагментация:** `uniform single random-cut`.

**Сценарий фиксированных кратностей ПЦР:** ПЦР1 увеличивает общее число молекул ровно в 20 раз, ПЦР2 — в 3,5 раза; новые копии раздаются случайно пропорционально числу копий. Каждая молекула порции фрагментируется отдельно. Бюджета ридов нет: каждая копия, отобранная на секвенирование, даёт одну пару.


## Структура выходных данных

`results/ERP003950/simulated/<ветка>/`, имя ветки собирается в ячейке параметров, например
`insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_ec1x3_rpc1_ss200-350h25_pcr1x20_pcr2x3.5_seq1_al1e-1`.

Внутри:
- `00_primary_truth/`
- `01_pcr1/`
- `02_fragmentation/`
- `03_pcr2/`
- `04_read_allocation/`
- `05_fastq_native/`
- `06_fastq_pe150/`
- `logs/`
- `qc/`

Для FastQC/MultiQC общий `qc.ipynb` использует `DATASET="ERP003950"`, `STAGE="simulated"` и имя ветки.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next(
    (p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))),
    None,
)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"

for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss",):
    path = shutil.which(tool)
    if not path:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {path}")

print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut"

# Эталон, входная порция библиотеки и глубина (входят в имя ветки)
# amplicon_with_primers: физический ампликон PCR1 = склеенный рид (V-праймер
#   в нём сохранён, остаток до V, V…J, C-фрагмент) + revcomp(C-праймер IgG).
# vj_only: эталон обрезан ровно до V…J.
TRUTH_TEMPLATE_MODE = "amplicon_with_primers"
MOUSE_C_PRIMER = "CARKGGATRRRCHGATGGGG"
ERROR_COLLAPSE_MAX_MISMATCHES = 1
ERROR_COLLAPSE_MIN_RATIO = 3
VERIFIED_MIN_READS = 2
# Бюджета пар нет: каждая копия, отобранная из пула ПЦР2 на секвенирование, даёт одну пару ридов.
READ_PAIRS_PER_SELECTED_COPY = 1

# Отбор по длине после фрагментации: задаётся явным диапазоном ниже.
# Внутри диапазона вероятность отбора 1; вне него она уменьшается вдвое на каждые h нт.
SIZE_SELECTION_RANGE = (200, 350)
SIZE_SELECTION_HALF_DISTANCE_NT = 25.0
# ПЦР без циклов и эффективности: общее число молекул растёт ровно в заданное число раз,
# новые копии раздаются случайно пропорционально числу копий шаблона (ПЦР1) или вида куска (ПЦР2).
PCR1_FOLD = 20
PCR2_FOLD = 3.5
PCR2_TO_SEQUENCING_FRACTION = 1.0  # на секвенирование идёт весь пул ПЦР2: при кратностях ×20 и ×3,5 его размер сопоставим с глубиной остальных сценариев
# Доля копий PCR1, случайно отбираемая в точном общем количестве.
PCR1_TO_FRAGMENTATION_FRACTION = 0.1  # фиксированная доля PCR1 для фрагментации

_TRUTH_TAG = {"amplicon_with_primers": "amp", "vj_only": "vj"}[TRUTH_TEMPLATE_MODE]
_TRUTH_TAG += f"_ec{ERROR_COLLAPSE_MAX_MISMATCHES}x{ERROR_COLLAPSE_MIN_RATIO:g}"
BRANCH_NAME = f"{BRANCH_NAME}_{_TRUTH_TAG}_rpc{READ_PAIRS_PER_SELECTED_COPY}_ss{SIZE_SELECTION_RANGE[0]}-{SIZE_SELECTION_RANGE[1]}h{SIZE_SELECTION_HALF_DISTANCE_NT:g}"
# Кратности ПЦР и доли библиотеки входят в имя ветки.
BRANCH_NAME += f"_pcr1x{PCR1_FOLD:g}_pcr2x{PCR2_FOLD:g}_seq{PCR2_TO_SEQUENCING_FRACTION:g}_al1e-1"
SOURCE_STAGE = "post_annotation_filtered"

RESULT_DATASET = "ERP003950"
RAW_DATASET = "ERP003950"

SAMPLES = [
    "ERR346596",
    "ERR346597",
    "ERR346598",
    "ERR346599",
    "ERR346600",
    "ERR346601",
]

RUN_LOCUS = {sample: "IGH" for sample in SAMPLES}

def resolve_volume():
    candidates = []

    env_root = os.environ.get("BCR_VOLUME")
    if env_root:
        candidates.append(Path(env_root))

    candidates.extend([
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ])

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    seen = set()

    for root in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)

        if (
            (root / "results" / RESULT_DATASET).is_dir()
            and (root / "raw" / RAW_DATASET).is_dir()
        ):
            return root

    raise FileNotFoundError(
        f"Cannot locate results/{RESULT_DATASET} together with raw/{RAW_DATASET}. "
        "Set BCR_VOLUME explicitly."
    )

VOLUME = resolve_volume()
DATASET_DIR = VOLUME / "results" / RESULT_DATASET

POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / RAW_DATASET

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run the fastp_q30_u40 mouse post-annotation filter first."
    )

FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == RESULT_DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS

MIN_VJ_SPAN_NT = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY_PERCENT = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY_PERCENT = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT_EVALUE = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT_EVALUE = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH_NT = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"


for directory in (
    TRUTH_DIR,
    PCR1_DIR,
    FRAGMENTATION_DIR,
    PCR2_DIR,
    ALLOCATION_DIR,
    FASTQ_NATIVE_DIR,
    FASTQ_DIR,
    MODEL_DIR,
    LOGS_DIR,
    QC_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

ISS_GENERATOR_CPUS = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False

DROP_SEQUENCES_WITH_N = True
# Исходные копии шаблона = его риды после схлопывания ошибок (UMI нет).
STARTING_COPIES_MODE = "observed_multiplicity"


# Фрагментация и отбор библиотеки.
# Из пула PCR1 случайно выбирается заданная доля копий; каждая выбранная
# молекула разрезается на две части, суммарная длина которых равна длине шаблона.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "uniform_single_cut"


# Отбор по длине после разрезания: диапазон SIZE_SELECTION_RANGE либо окно
# InSilicoSeq требует фрагмент длиннее рида модели NovaSeq (151 нт).
MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT = 152

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "pcr1_aliquot": "fixed-size random sample without replacement; exact total is round(fraction × PCR1 pool)",
    "pcr1_aliquot_fraction": PCR1_TO_FRAGMENTATION_FRACTION,
    "fragment_pool": "every aliquot molecule is cut individually; each daughter kept with p(length); identical (template, start, end) pieces pooled into one species",
    "pcr_model": "fixed total fold; new copies assigned at random proportional to current copies",
    "pcr1_fold": PCR1_FOLD,
    "pcr2_fold": PCR2_FOLD,
    "pcr2_sequencing_sample": "fixed-size random sample without replacement; exact total is round(fraction × PCR2 pool)",
    "pcr2_sequencing_fraction": PCR2_TO_SEQUENCING_FRACTION,
    "breakpoints": "exactly one uniformly sampled inter-base bond per library-input molecule",
    "daughter_fragments": "both physical daughters are created before size selection",
    "size_selection": {
        "range_nt": SIZE_SELECTION_RANGE,
        "half_distance_nt": SIZE_SELECTION_HALF_DISTANCE_NT,
        "min_sequencable_nt": MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT,
    },
}



SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("VOLUME:", VOLUME)
print("RESULT_DATASET:", RESULT_DATASET)
print("RAW_DATASET:", RAW_DATASET)
print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# Очистка существующих сгенерированных артефактов
# Удаляться могут только файлы внутри этой ветки симуляции.
# Исходные raw, filtered и annotation данные не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [
        TRUTH_DIR,
        PCR1_DIR,
        FRAGMENTATION_DIR,
        PCR2_DIR,
        ALLOCATION_DIR,
        FASTQ_NATIVE_DIR,
        FASTQ_DIR,
        MODEL_DIR,
        QC_DIR,
        LOGS_DIR,
    ]

    for directory in generated_dirs:
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        directory.mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение набора истины: ампликон PCR1

`post_annotation_filtered/airr_pass` является источником данных (читается `.airr.tsv` или `.airr.tsv.gz`).

Для каждого sample ноутбук:
1. повторно проверяет условия фильтра с помощью assertions;
2. схлопывает точные наблюдаемые V…J-последовательности (`v_sequence_start:j_sequence_end`) внутри sample; SHM сохраняются;
3. исключает записи с неоднозначными N (в режиме ампликона — во всём склеенном риде);
4. при `TRUTH_TEMPLATE_MODE = "amplicon_with_primers"` строит физический ампликон PCR1: самые частые для этого V…J фланги склеенного рида (V-праймер уже сохранён в риде, C-фрагмент) + revcomp C-праймера IgG, вырезанного cutadapt. Координаты V…J записаны в `vj_start`/`vj_end` (0-based, half-open) — этот интервал является эталоном для оценки сборщиков;
5. схлопывает ошибки секвенирования исходных ридов: вариант поглощается соседом той же длины, отличающимся одной заменой и встречающимся ≥ `ERROR_COLLAPSE_MIN_RATIO` раз чаще; его риды добавляются к соседу;
6. число исходных копий шаблона = его риды после схлопывания (UMI нет);
7. шаблон **проверен** (`verified`), если после схлопывания у него ≥ `VERIFIED_MIN_READS` ридов; непроверенные шаблоны симулируются, но не входят в оценку сборщиков. CDR3 (`junction`) записывается в `template_qc.tsv`.


In [ ]:
def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Коды неоднозначных нуклеотидов IUPAC; ниже каждый такой символ заменяется одной допустимой буквой.
_IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT",
          "K": "GT", "M": "AC", "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG", "N": "ACGT"}
# Таблица комплементарности оснований для построения обратного комплемента.
_RC = str.maketrans("ACGT", "TGCA")

def _revcomp(s):
    return s.translate(_RC)[::-1]

def _truth_airr_path(sample):
    for name in (f"{sample}.airr.tsv", f"{sample}.airr.tsv.gz"):
        if (TRUTH_AIRR_DIR / name).exists():
            return TRUTH_AIRR_DIR / name
    raise FileNotFoundError(f"Filtered AIRR missing: {TRUTH_AIRR_DIR}/{sample}.airr.tsv[.gz]")

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(row, sample):
    problems = []

    v_start = _as_int(row.get("v_sequence_start"))
    j_end = _as_int(row.get("j_sequence_end"))
    v_germline_start = _as_int(row.get("v_germline_start"))
    v_identity = _as_float(row.get("v_identity"))
    j_identity = _as_float(row.get("j_identity"))
    v_support = _as_float(row.get("v_support"))
    j_support = _as_float(row.get("j_support"))

    if (
        v_start is None
        or j_end is None
        or j_end - v_start + 1 < MIN_VJ_SPAN_NT
    ):
        problems.append("V-J span")

    if v_germline_start != 1:
        problems.append("V 5-prime completeness")

    if not _truthy(row.get("complete_vdj")):
        problems.append("complete_vdj")

    if str(row.get("locus", "")).strip() != RUN_LOCUS[sample]:
        problems.append("locus")

    if not str(row.get("v_call", "")).strip():
        problems.append("v_call")
    if not str(row.get("j_call", "")).strip():
        problems.append("j_call")

    if not _truthy(row.get("productive")):
        problems.append("productive")

    if not _falsey(row.get("stop_codon")):
        problems.append("stop_codon")

    if v_identity is None or v_identity < MIN_V_IDENTITY_PERCENT:
        problems.append("v_identity")
    if j_identity is None or j_identity < MIN_J_IDENTITY_PERCENT:
        problems.append("j_identity")
    if v_support is None or v_support > MAX_V_SUPPORT_EVALUE:
        problems.append("v_support")
    if j_support is None or j_support > MAX_J_SUPPORT_EVALUE:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{sample} airr_pass violates filter_summary contract for "
            f"{row.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    return {
        "dataset": RESULT_DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "truth_template_mode": TRUTH_TEMPLATE_MODE,
            "c_primer": MOUSE_C_PRIMER if TRUTH_TEMPLATE_MODE == "amplicon_with_primers" else None,
            "flanks": "most frequent (leader incl. retained V primer, C stub) among member reads + revcomp(C primer)",
            "vj_coordinates": "template_qc.tsv vj_start/vj_end, 0-based half-open",
            "collapse_scope": "within_sample",
            "collapse_key": ["VJ_sequence"],
            "starting_copies_mode": STARTING_COPIES_MODE,
            "dataset_has_umi": False,
            "error_collapse": {"max_mismatches": ERROR_COLLAPSE_MAX_MISMATCHES,
                               "min_parent_ratio": ERROR_COLLAPSE_MIN_RATIO,
                               "scope": "within_sample, same length, one level, directional"},
            "verified": f"reads after collapse >= {VERIFIED_MIN_READS}",
            "junction": "most frequent AIRR junction span among member reads",
        },
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
        "branch": BRANCH_NAME,
        "read_pairs_per_selected_copy": READ_PAIRS_PER_SELECTED_COPY,
        "pcr1_fold": PCR1_FOLD,
        "pcr2_fold": PCR2_FOLD,
        "seed": SEED,
    }

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False

    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and "
            "CLEAN_OLD_OUTPUTS=True, or choose a new branch."
        )
    return True

def _one_mismatch(a, b):
    d = 0
    for x, y in zip(a, b):
        if x != y:
            d += 1
            if d > 1:
                return False
    return True

def collapse_errors(counts):
    """child -> parent: вариант V…J поглощается соседом той же длины, отличающимся ровно
    одной заменой и встречающимся >= ERROR_COLLAPSE_MIN_RATIO раз чаще (самым частым из них).

    Кандидаты ищутся по половинам последовательности: при одной замене одна половина
    совпадает точно. Варианты обходятся от частых к редким; поглощённый вариант сам
    родителем не становится (одноуровневое, направленное схлопывание).
    """
    assert ERROR_COLLAPSE_MAX_MISMATCHES == 1
    index, parent = defaultdict(list), {}
    for s in sorted(counts, key=lambda x: (-counts[x], x)):
        L, h, n = len(s), len(s) // 2, counts[s]
        best = None
        for key, lo, hi in (((L, 0, s[:h]), h, L), ((L, 1, s[h:]), 0, h)):
            for p in index.get(key, ()):
                if (counts[p] >= ERROR_COLLAPSE_MIN_RATIO * n and _one_mismatch(s[lo:hi], p[lo:hi])
                        and (best is None or counts[p] > counts[best])):
                    best = p
        if best is not None:
            parent[s] = best
        elif n >= ERROR_COLLAPSE_MIN_RATIO:
            index[(L, 0, s[:h])].append(s)
            index[(L, 1, s[h:])].append(s)
    return parent

# самопроверка: 3 рида при соседе с 9 поглощаются; сосед с 4 < 3·9 не поглощает 9;
# вариант в 2 заменах от родителя остаётся; поглощённый вариант не становится родителем.
assert collapse_errors({"AAAAAAAA": 9, "AAAAAAAT": 3, "AAAAAATT": 1, "CAAAAAAA": 4}) == {"AAAAAAAT": "AAAAAAAA"}

def build_sample_templates(sample, force=FORCE):
    out_fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    out_tsv = TRUTH_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / f"{sample}_source_truth_audit.tsv"

    if (
        out_fasta.exists()
        and out_tsv.exists()
        and audit_path.exists()
        and not force
    ):
        _check_cached_truth_provenance()
        print(f"[{sample}] [skip] post-filter truth exists")
        return

    airr = _truth_airr_path(sample)
    filtered_fastq = FILTERED_FASTQ_DIR / f"{sample}_filtered.fastq.gz"

    if not filtered_fastq.exists():
        raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")

    expected_passed = int(FILTER_SUMMARY["samples"][sample]["passed"])

    counts = defaultdict(int)
    observed = defaultdict(int)
    flanks = defaultdict(lambda: defaultdict(int))
    junctions = defaultdict(Counter)

    with open_text(airr) as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        for row in reader:
            counts["airr_pass_rows"] += 1
            _validate_post_filter_row(row, sample)

            start = int(row["v_sequence_start"]) - 1
            end = int(row["j_sequence_end"])
            full = (row.get("sequence") or "").upper()
            seq = full[start:end]

            if len(seq) < MIN_VJ_SPAN_NT:
                raise RuntimeError(
                    f"{sample}: cropped V-J sequence shorter than filter threshold "
                    f"for {row.get('sequence_id')}"
                )

            if DROP_SEQUENCES_WITH_N and "N" in (full if TRUTH_TEMPLATE_MODE == "amplicon_with_primers" else seq):
                counts["simulation_excluded_N"] += 1
                continue

            observed[seq] += 1
            flanks[seq][(full[:start], full[end:])] += 1
            # junction = sequence[cdr3_start-4 : cdr3_end+3] (AIRR, координаты с 1), в координатах V…J.
            if row.get("junction") and row.get("cdr3_start"):
                junctions[seq][(int(row["cdr3_start"]) - 4 - start, len(row["junction"]))] += 1
            counts["simulation_eligible_rows"] += 1

    if counts["airr_pass_rows"] != expected_passed:
        raise RuntimeError(
            f"{sample}: airr_pass rows={counts['airr_pass_rows']:,}, "
            f"but filter_summary passed={expected_passed:,}"
        )

    # Схлопывание ошибок секвенирования: риды поглощённого варианта уходят соседу.
    parent_of = collapse_errors(observed)
    copies = dict(observed)
    absorbed = defaultdict(int)
    for child, parent in parent_of.items():
        copies[parent] += observed[child]
        absorbed[parent] += 1
        del copies[child]

    ordered = sorted(
        copies.items(),
        key=lambda item: (-item[1], item[0]),
    )

    iupac_rng = np.random.default_rng(sample_seed(sample, 500))
    amplicon = TRUTH_TEMPLATE_MODE == "amplicon_with_primers"

    def _junction(seq, spans):
        if not spans:
            return ""
        jo, jl = spans.most_common(1)[0][0]
        return seq[jo:jo + jl] if 0 <= jo and jo + jl <= len(seq) else ""

    with open(out_fasta, "w") as fasta, open(out_tsv, "w", newline="") as table:
        fields = [
            "template_id",
            "locus",
            "length",
            "vj_start",
            "vj_end",
            "observed_multiplicity",
            "exact_reads",
            "absorbed_variants",
            "verified",
            "junction",
        ]
        writer = csv.DictWriter(table, fieldnames=fields, delimiter="\t")
        writer.writeheader()

        for i, (seq, multiplicity) in enumerate(ordered, 1):
            template_id = f"{sample}_IGH_tpl_{i:07d}"
            if amplicon:
                # самые частые фланги среди ридов этого V…J; вырожденные позиции
                # праймера разрешаются воспроизводимо
                lead, cstub = max(flanks[seq].items(), key=lambda kv: kv[1])[0]
                c_primer = "".join(
                    c if c in "ACGT" else str(iupac_rng.choice(list(_IUPAC[c])))
                    for c in MOUSE_C_PRIMER
                )
                template = lead + seq + cstub + _revcomp(c_primer)
            else:
                lead, template = "", seq
            fasta.write(f">{template_id}\n{template}\n")
            writer.writerow({
                "template_id": template_id,
                "locus": "IGH",
                "length": len(template),
                "vj_start": len(lead),
                "vj_end": len(lead) + len(seq),
                "observed_multiplicity": multiplicity,
                "exact_reads": observed[seq],
                "absorbed_variants": absorbed[seq],
                "verified": multiplicity >= VERIFIED_MIN_READS,
                "junction": _junction(seq, junctions[seq]),
            })
    audit = {
        "sample": sample,
        "expected_passed_from_filter_summary": expected_passed,
        "airr_pass_rows": counts["airr_pass_rows"],
        "simulation_eligible_rows": counts["simulation_eligible_rows"],
        "simulation_excluded_N": counts["simulation_excluded_N"],
        "unique_exact_vj": len(observed),
        "absorbed_error_variants": len(parent_of),
        "templates_after_collapse": len(ordered),
        "verified_templates": sum(n >= VERIFIED_MIN_READS for n in copies.values()),
    }
    with open(audit_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(audit), delimiter="\t")
        writer.writeheader()
        writer.writerow(audit)

    print(
        f"[{sample}] airr_pass={counts['airr_pass_rows']:,}; "
        f"simulation_eligible={counts['simulation_eligible_rows']:,}; "
        f"unique_VJ={len(observed):,}; after_collapse={len(ordered):,}; "
        f"excluded_N={counts['simulation_excluded_N']:,}"
    )

for sample in SAMPLES:
    build_sample_templates(sample)

PROVENANCE_PATH.write_text(
    json.dumps(_current_provenance(), indent=2) + "\n"
)
print("provenance:", PROVENANCE_PATH)


## 5. Сводка QC точного набора истины V…J


In [ ]:
template_summary = []

for sample in SAMPLES:
    path = TRUTH_DIR / f"{sample}_template_qc.tsv"
    rows = list(csv.DictReader(open(path), delimiter="\t"))
    multiplicities = np.asarray(
        [int(row["observed_multiplicity"]) for row in rows],
        dtype=np.int64,
    )

    template_summary.append({
        "sample": sample,
        "locus": "IGH",
        "unique_templates": len(rows),
        "filtered_reads_represented": int(multiplicities.sum()),
        "singleton_templates": int((multiplicities == 1).sum()),
        "verified_templates": sum(row["verified"] == "True" for row in rows),
        "max_observed_multiplicity": (
            int(multiplicities.max()) if len(multiplicities) else 0
        ),
    })

out = QC_DIR / "template_summary.tsv"
with open(out, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(template_summary[0]),
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(template_summary)

print(*template_summary, sep="\n")
print("wrote", out)


## 6. Встроенная модель ошибок NovaSeq из InSilicoSeq

Эта ветка не обучает эмпирическую модель ошибок на ERP003950. Используется общий
встроенный профиль NovaSeq, что делает слой ошибок секвенирования сопоставимым между датасетами.


In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH_NT})


In [ ]:
# В этой ветке модель не обучается: используется встроенный KDE-профиль NovaSeq.

In [ ]:
# Происхождение модели записывается в ячейке параметров выше.

## 7. Число ридов

Бюджета пар нет: каждая копия, отобранная из пула ПЦР2 на секвенирование, даёт одну пару ридов PE150. Число пар каждого образца становится известно после раздела 11.


## 8. PCR1 для точных V…J templates после фильтрации


In [ ]:
def load_dedup_templates(sample):
    fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    qc = TRUTH_DIR / f"{sample}_template_qc.tsv"
    meta = {row["template_id"]: row for row in csv.DictReader(open(qc), delimiter="\t")}
    rows = []
    for template_id, seq in iter_fasta(fasta):
        m = meta[template_id]
        rows.append({
            "template_id": template_id, "sequence": seq, "length": len(seq),
            "vj_start": int(m.get("vj_start") or 0),
            "vj_end": int(m.get("vj_end") or len(seq)),
            "locus": m["locus"],
            "observed_multiplicity": int(m["observed_multiplicity"]),
            "verified": m["verified"] == "True",
        })
    return rows


def starting_copies(row):
    if STARTING_COPIES_MODE == "one_per_unique":
        return 1
    if STARTING_COPIES_MODE == "observed_multiplicity":
        return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)


def fixed_fold_amplification(copies, fold, rng):
    """Увеличивает общее число копий пула ровно в fold раз (с округлением до целого).

    Каждая новая копия достаётся случайной молекуле пула: вероятность получить её
    пропорциональна текущему числу копий записи (шаблона или вида куска).
    """
    copies = np.asarray(copies, dtype=np.int64)
    total = int(copies.sum())
    new_total = int(np.floor(total * fold + 0.5)) - total
    if total == 0 or new_total <= 0:
        return copies.copy()
    owners = np.searchsorted(
        np.cumsum(copies), rng.integers(0, total, size=new_total), side="right"
    )
    return copies + np.bincount(owners, minlength=len(copies))


def simulate_pcr_pool(sample):
    rng = np.random.default_rng(sample_seed(sample, 1000))
    rows = load_dedup_templates(sample)
    start = np.array([starting_copies(row) for row in rows], dtype=np.int64)
    pcr_copies = fixed_fold_amplification(start, PCR1_FOLD, rng)
    for row, s, copies in zip(rows, start, pcr_copies):
        row.update(starting_copies=int(s), pcr_copies=int(copies))
    return rows


_t = fixed_fold_amplification(np.array([1, 10, 0]), 20, np.random.default_rng(1))
assert int(_t.sum()) == 220 and _t[2] == 0 and _t[0] >= 1 and _t[1] >= 10
_t = fixed_fold_amplification(np.ones(10000, dtype=np.int64), 3.5, np.random.default_rng(1))
assert int(_t.sum()) == 35000 and _t.min() >= 1
print("fixed-fold PCR self-check: OK")


## 9. Фрагментация и отбор по длине

1. Из пула ПЦР1 случайно без возвращения отбирается ровно round(N × PCR1_TO_FRAGMENTATION_FRACTION) молекул.
2. Каждая молекула порции рвётся отдельно по одной из L−1 связей, все связи равновероятны: получаются куски [0, c) и [c, L).
3. Каждый кусок независимо проходит отбор по длине: внутри SIZE_SELECTION_RANGE вероятность 1, вне него 2^(−d/h); куски короче 152 нт не проходят.
4. Одинаковые куски (тот же шаблон, те же start и end) от разных молекул объединяются в один вид куска с числом копий k ≥ 1.


In [ ]:
def _size_selection_probability(length):
    """Возвращает вероятность отбора фрагмента по длине для ПЦР2."""
    length = np.asarray(length, dtype=float)
    low, high = SIZE_SELECTION_RANGE
    distance = np.maximum(np.maximum(low - length, length - high), 0.0)
    probability = 2.0 ** (-distance / SIZE_SELECTION_HALF_DISTANCE_NT)
    return np.where(length < MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT, 0.0, probability)


FRAGMENT_ID_TAG = "rc"


def fragment_molecules(molecule_template, molecule_length, rng):
    """Поштучная фрагментация молекул порции и отбор кусков по длине.

    Молекула длины L рвётся по одной из L−1 связей, все связи равновероятны;
    получаются куски [0, c) и [c, L). Каждый кусок независимо проходит отбор
    с вероятностью p(длины). Возвращает шаблон, start и end прошедших кусков.
    """
    molecule_length = np.asarray(molecule_length, dtype=np.int64)
    cut = 1 + (rng.random(len(molecule_length)) * (molecule_length - 1)).astype(np.int64)
    template = np.concatenate([molecule_template, molecule_template])
    start = np.concatenate([np.zeros_like(cut), cut])
    end = np.concatenate([cut, molecule_length])
    keep = rng.random(len(start)) < _size_selection_probability(end - start)
    return template[keep], start[keep], end[keep]


# самопроверка: разрывы внутри молекулы, доля прошедших кусков совпадает с ожидаемой
_L, _n = 450, 200000
_t, _s, _e = fragment_molecules(np.zeros(_n, dtype=np.int64), np.full(_n, _L), np.random.default_rng(1))
_expected = 2 * float(_size_selection_probability(np.arange(1, _L)).mean())
assert _s.min() >= 0 and _e.max() <= _L and ((_e - _s) >= MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT).all()
assert abs(len(_s) / _n - _expected) < 0.01
print("fragmentation self-check: OK")


## 10. ПЦР2 и отбор на секвенирование

Общее число кусков после отбора по длине увеличивается ровно в PCR2_FOLD раз: новые копии раздаются случайно пропорционально числу копий каждого вида куска (та же функция `fixed_fold_amplification`, что в ПЦР1). Затем из пула ПЦР2 случайно без возвращения отбирается ровно round(N × PCR2_TO_SEQUENCING_FRACTION) копий; каждая отобранная копия даёт одну пару ридов.


## 11. PCR1 → порция → фрагментация → PCR2 → распределение ридов


In [ ]:
def sample_fixed_size_without_replacement(counts, sample_size, rng):
    """Случайно выбирает заданное число копий из группированного пула.

    Счётчики объединяют копии по шаблонам или видам фрагментов. Сначала
    строится биномиальное приближение, затем случайно корректируется избыток
    или нехватка выбранных копий. В результате получается выборка без
    возвращения с многомерным гипергеометрическим распределением; ограничение
    NumPy для очень больших пулов не используется.
    """
    counts = np.asarray(counts, dtype=np.int64)
    if np.any(counts < 0):
        raise ValueError("Число копий в пуле не может быть отрицательным")
    total = sum(int(value) for value in counts)
    sample_size = int(sample_size)
    if sample_size < 0 or sample_size > total:
        raise ValueError(f"Размер выборки {sample_size} вне диапазона [0, {total}]")
    if sample_size == 0:
        return np.zeros_like(counts)
    if sample_size == total:
        return counts.copy()
    if sample_size > total - sample_size:
        return counts - sample_fixed_size_without_replacement(
            counts, total - sample_size, rng
        )

    probability = sample_size / total
    proposal = np.asarray(rng.binomial(counts, probability), dtype=np.int64)
    proposed_total = sum(int(value) for value in proposal)
    if proposed_total > sample_size:
        correction = sample_fixed_size_without_replacement(
            proposal, proposed_total - sample_size, rng
        )
        return proposal - correction
    if proposed_total < sample_size:
        failures = counts - proposal
        correction = sample_fixed_size_without_replacement(
            failures, sample_size - proposed_total, rng
        )
        return proposal + correction
    return proposal


def sample_fraction_without_replacement(counts, fraction, rng):
    """Случайно выбирает из пула точно заданную долю копий."""
    total = sum(int(value) for value in counts)
    if not 0 <= fraction <= 1:
        raise ValueError("Доля выборки должна быть от 0 до 1")
    target = int(np.floor(total * fraction + 0.5))
    return sample_fixed_size_without_replacement(counts, target, rng)


def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows({k: r[k] for k in fields} for r in rows)
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR1", out)
        return
    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "starting_copies", "pcr_copies",
    ]
    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)


def run_library_stage(sample, force=FORCE):
    """Порция ПЦР1 → поштучная фрагментация и отбор → ПЦР2 → доля на секвенирование → риды."""
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_library_summary.tsv"
    if all(p.exists() for p in (out_counts, out_fa, out_qc, summary_out)) and not force:
        print("[skip] library", sample)
        return

    rows = read_dict_rows(inp)
    seqs = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    rng = np.random.default_rng(sample_seed(sample, 2000))
    pcr1 = np.array([int(row["pcr_copies"]) for row in rows], dtype=np.int64)
    lengths = np.array([int(row["length"]) for row in rows], dtype=np.int64)

    # порция ПЦР1: точная случайная доля пула без возвращения
    aliquot = sample_fraction_without_replacement(pcr1, PCR1_TO_FRAGMENTATION_FRACTION, rng)
    molecule_template = np.repeat(np.arange(len(rows)), aliquot)
    piece_template, piece_start, piece_end = fragment_molecules(
        molecule_template, lengths[molecule_template], rng
    )

    # одинаковые куски разных молекул — один вид куска с числом копий k
    span = int(lengths.max()) + 1
    species, input_copies = np.unique(
        (piece_template * span + piece_start) * span + piece_end, return_counts=True
    )
    species_template = species // (span * span)
    species_start = (species // span) % span
    species_end = species % span

    pcr2 = fixed_fold_amplification(input_copies, PCR2_FOLD, rng)
    selected = sample_fraction_without_replacement(pcr2, PCR2_TO_SEQUENCING_FRACTION, rng)
    pairs = selected * READ_PAIRS_PER_SELECTED_COPY

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "fragment_input_copies",
        "library_pcr2_copies", "sequencing_selected_copies", "simulated_read_pairs",
    ]
    tmp = {path: Path(str(path) + ".tmp") for path in (out_counts, out_fa, out_qc)}
    with open(tmp[out_counts], "w", newline="") as counts_handle, \
         open(tmp[out_fa], "w") as fasta_handle, \
         open(tmp[out_qc], "w", newline="") as qc_handle:
        counts_writer = csv.writer(counts_handle, delimiter="\t")
        qc_writer = csv.DictWriter(qc_handle, fieldnames=fields, delimiter="\t")
        qc_writer.writeheader()
        for j in np.flatnonzero(pairs):
            row = rows[species_template[j]]
            start, end = int(species_start[j]), int(species_end[j])
            fragment_id = f"{row['template_id']}_{FRAGMENT_ID_TAG}_{start}_{end}"
            # в readcount_file InSilicoSeq передаётся число ридов, а не пар
            counts_writer.writerow([fragment_id, int(pairs[j]) * 2])
            fasta_handle.write(f">{fragment_id}\n{seqs[row['template_id']][start:end]}\n")
            qc_writer.writerow({
                "fragment_id": fragment_id,
                "template_id": row["template_id"],
                "locus": row["locus"],
                "template_length": int(lengths[species_template[j]]),
                "fragment_start": start,
                "fragment_length": end - start,
                "fragment_input_copies": int(input_copies[j]),
                "library_pcr2_copies": int(pcr2[j]),
                "sequencing_selected_copies": int(selected[j]),
                "simulated_read_pairs": int(pairs[j]),
            })
    for final, temporary in tmp.items():
        temporary.replace(final)

    n_input = int(aliquot.sum())
    with_reads = pairs > 0
    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_templates": len(rows),
        "starting_copies_total": sum(int(row["starting_copies"]) for row in rows),
        "pcr1_fold": PCR1_FOLD,
        "pcr1_molecules_total": int(pcr1.sum()),
        "library_input_molecules": n_input,
        "library_aliquot_fraction_of_pcr1": round(n_input / int(pcr1.sum()), 6),
        "library_input_templates": int(np.count_nonzero(aliquot)),
        "fragmentation_pieces_total": 2 * n_input,
        "fragments_after_size_selection": int(len(piece_start)),
        "retained_fragments_per_input_molecule": round(len(piece_start) / n_input, 4) if n_input else 0,
        "fragment_species": int(len(species)),
        "fragment_species_with_duplicates": int((input_copies > 1).sum()),
        "pcr2_fold": PCR2_FOLD,
        "pcr2_pool_copies": int(pcr2.sum()),
        "pcr2_selected_copies": int(selected.sum()),
        "pcr2_selected_fraction": int(selected.sum()) / int(pcr2.sum()) if int(pcr2.sum()) else 0,
        "read_pairs": int(pairs.sum()),
        "templates_with_reads": int(len(np.unique(species_template[with_reads]))),
        "fragment_species_with_reads": int(with_reads.sum()),
        "size_selection_range_low_nt": SIZE_SELECTION_RANGE[0],
        "size_selection_range_high_nt": SIZE_SELECTION_RANGE[1],
        "size_selection_half_distance_nt": SIZE_SELECTION_HALF_DISTANCE_NT,
        "min_sequencable_fragment_nt": MIN_FRAGMENT_LENGTH_FOR_SEQUENCING_NT,
    }
    write_dict_rows(summary_out, [summary], list(summary))
    print(f"[{sample}] library: {summary}")


for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_library_stage(sample)


## 12. Сводка библиотеки


In [ ]:
library_summary = [read_dict_rows(FRAGMENTATION_DIR / f"{sample}_library_summary.tsv")[0] for sample in SAMPLES]
out = QC_DIR / "library_summary.tsv"
write_dict_rows(out, library_summary, list(library_summary[0]))
print(*library_summary, sep="\n")
EXPECTED_PAIRS = {row["sample"]: int(row["read_pairs"]) for row in library_summary}


## 12a. Восстановимость эталонных шаблонов (reconstructability)

Для каждого эталонного шаблона по фрагментам, получившим риды, определяется, что сборщик **в принципе** может восстановить в интервале V…J:

| класс | смысл |
|---|---|
| `no_reads` | V…J шаблона не попал в FASTQ |
| `partial_read_coverage` | часть V…J не прочитана ни одним ридом |
| `full_coverage_unlinked` | всё прочитано, но есть точка, которую не пересекает ни один фрагмент: куски физически не связаны |
| `full_coverage_mate_linked_only` | каждая точка пересечена фрагментом, но хотя бы одна — только промежутком между R1 и R2 (mate-pair extension) |
| `full_coverage_read_overlap` | каждая точка внутри рида с `RECONSTRUCT_MIN_FLANK_NT` с обеих сторон — сборка обычным перекрытием |

Эти классы — знаменатели для benchmark сборщиков.


In [ ]:
# Класс восстановимости эталонного интервала V…J каждого шаблона по тому, что реально попало в FASTQ.
# В режиме ампликона InSilicoSeq: R1 = [start, start+R), R2 = [end-R, end); вся вставка остаётся физически связанной.
# Связь (bond) b между позициями b-1 и b:
#   read_overlap — связь попадает в один рид, и с обеих сторон остаётся не менее K нт.
#                  (собирается обычным перекрытием);
#   mate_only — связь лежит во фрагменте, но между ридами, в несеквенированном промежутке.
#               соединить её может только удлинение по паре ридов в TRUST4.
#   unlinked — связь не пересекается ни одним фрагментом: части шаблона физически
#              не связаны и не могут быть соединены без внешней информации.
RECONSTRUCT_MIN_FLANK_NT = 31  # грубый аналог минимального перекрытия (k-мера) сборщика, калибруемый

def _bond_mask(L, intervals):
    d = np.zeros(L + 2, dtype=np.int32)
    for a, b in intervals:
        a, b = max(a, 1), min(b, L - 1)
        if a <= b:
            d[a] += 1
            d[b + 1] -= 1
    return np.cumsum(d)[: L + 1] > 0

def classify_template(L, frags, R=TARGET_READ_LENGTH_NT, K=RECONSTRUCT_MIN_FLANK_NT, region=None):
    """frags: [(start, end)] фрагментов с ридами в координатах шаблона.

    region=(vj_start, vj_end): оценивается только эталонный интервал V…J
    (праймеры и leader/C-фланги не требуют сборки). Возвращает (class, read_covered_fraction).
    """
    reads = [iv for s, e in frags for iv in ((s, min(s + R, e)), (max(e - R, s), e))]
    if region is not None:
        a0, b0 = region
        clip = lambda ivs: [(max(a, a0) - a0, min(b, b0) - a0) for a, b in ivs if min(b, b0) > max(a, a0)]
        L, frags, reads = b0 - a0, clip(frags), clip(reads)
    if not frags:
        return "no_reads", 0.0
    cov = np.zeros(L + 1, dtype=np.int32)
    for a, b in reads:
        cov[a] += 1
        cov[b] -= 1
    covered = float((np.cumsum(cov)[:L] > 0).mean())
    if covered < 1.0:
        return "partial_read_coverage", covered
    by_frag = _bond_mask(L, [(s + K, e - K) for s, e in frags])
    by_read = _bond_mask(L, [(a + K, b - K) for a, b in reads])
    bonds = slice(K, L - K + 1)  # связи ближе K к краям шаблона не требуют соединения
    if not by_frag[bonds].all():
        return "full_coverage_unlinked", covered
    if not by_read[bonds].all():
        return "full_coverage_mate_linked_only", covered
    return "full_coverage_read_overlap", covered

# Самопроверка: фрагмент 400 нт, парные риды PE150.
assert classify_template(400, [])[0] == "no_reads"
assert classify_template(400, [(0, 200), (200, 400)])[0] == "full_coverage_unlinked"
assert classify_template(300, [(0, 200), (100, 300)])[0] == "full_coverage_read_overlap"
assert classify_template(400, [(0, 400)])[0] == "partial_read_coverage"          # середина между R1 и R2 не прочитана
assert classify_template(300, [(0, 300)])[0] == "full_coverage_mate_linked_only"  # R1 и R2 стыкуются без перекрытия
assert classify_template(420, [(0, 420), (100, 320)])[0] == "full_coverage_mate_linked_only"
# фланги вне V…J не учитываются; фрагмент только в праймере = нет ридов в V…J
assert classify_template(500, [(0, 300), (200, 500)], region=(50, 450))[0] == "full_coverage_mate_linked_only"
assert classify_template(500, [(0, 40)], region=(50, 450))[0] == "no_reads"

def write_reconstructability_qc(sample):
    frags = defaultdict(list)
    for r in read_dict_rows(ALLOCATION_DIR / f"{sample}_allocation.tsv"):
        if int(r["simulated_read_pairs"]) > 0:
            s = int(r["fragment_start"])
            frags[r["template_id"]].append((s, s + int(r["fragment_length"])))
    rows, summary = [], defaultdict(lambda: defaultdict(int))
    for t in load_dedup_templates(sample):
        cls, cov = classify_template(t["length"], frags.get(t["template_id"], []),
                                     region=(t["vj_start"], t["vj_end"]))
        rows.append({"template_id": t["template_id"], "locus": t["locus"], "length": t["length"],
                     "observed_multiplicity": t["observed_multiplicity"],
                     "fragments_with_reads": len(frags.get(t["template_id"], [])),
                     "read_covered_fraction": round(cov, 4), "reconstructability": cls, "verified": t["verified"]})
        if t["verified"]:  # знаменатели оценки сборщиков — только проверенные шаблоны
            summary[t["locus"]][cls] += 1
    write_dict_rows(QC_DIR / f"{sample}_template_reconstructability.tsv", rows, list(rows[0]))
    classes = ["no_reads", "partial_read_coverage", "full_coverage_unlinked",
               "full_coverage_mate_linked_only", "full_coverage_read_overlap"]
    out = []
    for locus in sorted(summary) + ["ALL"]:
        c = {k: (sum(summary[l][k] for l in summary) if locus == "ALL" else summary[locus][k]) for k in classes}
        n = sum(c.values())
        out.append({"sample": sample, "locus": locus, "templates": n, **c,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in classes}})
    write_dict_rows(QC_DIR / f"{sample}_reconstructability_summary.tsv", out, list(out[0]))
    print(*out, sep="\n")

for sample in SAMPLES:
    write_reconstructability_qc(sample)


## 13. Генерация ридов PE150 с профилем NovaSeq


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(ISS_GENERATOR_CPUS),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH_NT); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH_NT); assert n1==n2==EXPECTED_PAIRS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":EXPECTED_PAIRS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==EXPECTED_PAIRS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH_NT]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и ограничения

- Симуляция относится только к ветке предобработки `ERP003950`.
- Источник набора истины — собственный `post_annotation_filtered/airr_pass` этой ветки.
- В датасете нет UMI: ошибки исходных ридов схлопываются (1 замена, сосед ≥3× чаще), исходные копии = риды после схлопывания. Это включает PCR-искажения исходной библиотеки; шаблоны с 1 ридом в оценку не входят.
- Точная наблюдаемая V…J-последовательность сохраняет настоящую SHM; germline-последовательность не подставляется.
- Шаблон — физический ампликон PCR1 (склеенный рид с V-праймером + revcomp C-праймера); эталон для оценки сборки — интервал `vj_start…vj_end`. Вырожденные позиции C-праймера разрешены случайно, но воспроизводимо.
- Кратности ПЦР, доли аликвоты ПЦР1/ПЦР2, параметры схлопывания и число пар на отобранную копию входят в имя ветки.
- ПЦР1 и ПЦР2 заданы фиксированной кратностью общего числа молекул (×20 и ×3,5) без циклов и эффективности; новые копии раздаются случайно пропорционально числу копий. Ошибки полимеразы и химеры не моделируются.
- Для ошибок секвенирования используется встроенная модель InSilicoSeq NovaSeq. Нативные риды PE151 детерминированно обрезаются до PE150.
- Бюджета ридов нет: число пар равно числу копий, отобранных из пула ПЦР2 на секвенирование.
